## CNN avec tensorFlow

In [ ]:
# Import Librairy
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import regularizers
import numpy as np
import matplotlib.pyplot as plt #can be replace by plotly


### Import un dataset

Divisé en deux dataset plus petit : \n
- Train
- Test

Train sera utilisé pour entrainé le modele à reconnaitre les images (reconnaissance binaire ou d'un groupe d'objet), test sera la pour "valider" cet entrainement et voir si le modèle doit etre retoucher pour obtenir un meilleur résultat.
Partager le dataset par rapport au nombre d'image dans le dataset

In [ ]:
fashion_mnist = tf.keras.datasets.fashion_mnist #Dataset

(x_train, train_labels), (x_test, test_labels) = fashion_mnist.load_data()

### Shape
Les valeurs habituelle d'une image son entre 0 et 255, pour cela il faut normaliser cela en premier lieu divisant par 255 pour avoir des valeurs comprise entre 0 et 1. Car cela va aidé à la descente de gradient et rentre plus rapide l'entrainement du modele. Uniquement Pour X car y est généralement les labels de l'image si classification binaire, 1 pour l'image est une photo 0 pas une photo.

In [ ]:
x_train, x_test = x_train / 255.0, x_test / 255.0
x_train.shape, x_test.shape

((60000, 28, 28), (10000, 28, 28))

### EarlyCallback et backpropagation

- On utilise le EarlyCallback pour stopper l'entrainement du modèle quand la valeur cible ne s'améliore pas, on défini une patience qui s'y active après la valeur x défini est atteinte et que pendant cette période il y a eu aucune amélioration et limité l'overfitting. Restore_best_weights va restoré l'epoch avec la meilleure valeur obtenue quand le modèle va se stopper.

In [ ]:
early_stopping_callback = EarlyStopping(
    monitor='val_loss', # The metric to monitor for improvement
    patience=8,         # Number of epochs with no improvement after which training will be stopped
    verbose=1,          # Show a message when stopping
    restore_best_weights=True # Restores model weights from the epoch with the best value of the monitored quantity.
)

## Création Modele

En utilisant plusieurs couches dans le modèle pour le CNN, cela permet de trier et determiné l'ouput que l'on souhaite, dans les couches. On trouve :
- ***input_shape :*** est la taille d'entrée et le nombre channel de l'image et se défini comme (hauteur, largeur, channel). Vous pouvez rajouter aussi batch_size.
- ***Convolutional Layer:*** Chargé d'extraire des caractéristiques locales de l'image avec un kernel d'une taille défini par souvent de matrice 3x3 en passant sur toute l'image, cela permet d'extraire les informations de l'image, fonctionne avec Pooling.
- ***Pooling Layer:*** Le pooling layer fonctionne avec le Convolutional layer, il y a deux type de pooling soit average Pooling ou Max Pooling, ce layer permet de réduire le taille des caractèristiques obtenu par la couche Convolutional pour améliorer la vitesse d'entrainement, le plus souvent on utilise MaxPooling, souvent une fenetre entre 2x2 et 3x3 en prennant le maximun de la fenetre et mets en évidence les caractéristique détecté dans le layer précédent.
- ***Dense :*** Un Dense layer est un layer avec les neuronnes tous interconnecté, il est placé à la fin après flatten car il ne peut comprendre uniquement des vecteurs en une dimension. Pour la couche de sortie est dense, tu défini avec softmax ou sigmoid par rapport à ton besoin. Sigmoid = 1 dense layer de sortie car on l'utilise pour la classification binaire, Softmax = depend du nbr de classe à prédire pour la classification multi-classe.
- ***Flatten :*** Transforme les layers de plusieurs dimensions en un vecteur de 1 seule dimension pour que les Denses layers puissent traduire l'image.
- ***Dropout:*** Désactive certains neurones pour éviter l'overfitting, il faut souvent tester plusieurs valeurs comprise entre 0 et 1 pour obtenir le meilleur parametres mais le plus souvent on trouve une valeur entre 0.1 et 0.4.
- ***L2 regularization :*** Va appliqué une pénalité sur les poids trop grand, cela force pendant l'optimisation des poids se rapproche de 0, tu décides avec l'hyperparamètres lambda ($\lambda$) la puissance de cette pénalité.


In [ ]:
model = keras.Sequential([
    keras.Input(shape=(28, 28, 3)),
    
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.1),
    
    layers.Flatten(),
    layers.Dense(128, activation='relu',kernel_regularizer=regularizers.l2(0.001)),
    layers.Dense(10, activation='softmax',kernel_regularizer=regularizers.l2(0.001))
])

model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 26, 26, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 225,610 (881.29 KB)

 Trainable params: 225,610 (881.29 KB)

 Non-trainable params: 0 (0.00 B)

### Compilé et L'optimisateur

On utilise un optimiser pour mettre à jour les poids en utilisant la descente de gradient lors de la backpropagation, le plus souvent on utilise l'agorithme Adam qui mets à jour automatiquement le Learning rate pour chaque poids en améliorant la vitesse et les performances d'entrainement.
Lorsqu'on compile le modèle avec TensorFlow, on choisit la métrique que l'on peut observé, ici l'accuracy de notre modèle. 
La loss fonction est utilisé pour calculé la différence entre les prédictions de notre modèle et les véritables images. Plusieurs méthodes sont disponible pour le CNN:
- Binary Cross-Entropy (Pour la classification binaire)
- Categorical Cross-Entropy (Pour la classification de plusieurs classes)
- Sparse Categorical Crossentropy (Pour la classification de plusieurs classes, mais les labels sont des numéros entier 1,2 etc...)

In [ ]:
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

### Lancement du modèle
***Les epochs:*** sont le nombre de fois que le modèle s'entraine

***Validation Split:*** est le % du dataset utilisé pour valider l'entrainement.

***Model.fit:*** est le lancement de l'entrainement du modèle.

***Model.evaluate:*** Permet d'evaluer le model sur sa precision et si les résultats sont proche des vrai images avec le dataset de Teest.

In [ ]:
all_callbacks = [early_stopping_callback]
history =model.fit(x_train, train_labels, epochs=50,callbacks=all_callbacks, validation_split=0.1)

# Evaluate the model on the unseen test data
test_loss, test_accuracy = model.evaluate(x_test, test_labels)
print(f"\nTest Accuracy: {test_accuracy}")

### Graphique des résultats

In [ ]:
hist_dict = history.history

epochs = range(1, len(hist_dict['accuracy']) + 1)

plt.figure(figsize=(14, 5))

# 1. Graphique de la Précision (Accuracy)
plt.subplot(1, 2, 1) # (1 ligne, 2 colonnes, position 1)
plt.plot(epochs, hist_dict['accuracy'], label='Précision (Entraînement)', marker='o')
plt.plot(epochs, hist_dict['val_accuracy'], label='Précision (Validation)', marker='o')
plt.title('Évolution de la Précision au cours des Epochs')
plt.xlabel('Epoch')
plt.ylabel('Précision')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)

# 2. Graphique de la Perte (Loss)
plt.subplot(1, 2, 2) # (1 ligne, 2 colonnes, position 2)
plt.plot(epochs, hist_dict['loss'], label='Perte (Entraînement)', marker='o')
plt.plot(epochs, hist_dict['val_loss'], label='Perte (Validation)', marker='o')
plt.title('Évolution de la Perte au cours des Epochs')
plt.xlabel('Epoch')
plt.ylabel('Perte')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

### Export du modèle

In [ ]:
model.save("Modele_cnn.keras")